In [15]:
import os
import sys

if os.path.basename(os.getcwd()) == "analysis":
    os.chdir(os.path.dirname(os.getcwd()))
    sys.path.append(os.getcwd())

full_runs_dir = "../logs_cluster/logs/full_runs/february/"



In [16]:
import re
from pathlib import Path

# All requested runs under march
run_dirs = [
    "2026_01_15_YCoCg_two_by_two_Clic",
    "2026_01_30_YCoCg_one_by_one_Clic",
    "2026_01_30_YCoCg_tre_by_tre_Clic",
    "2026_02_09_YCoCg_tre_by_tre_Clic",

]

log_name_re = re.compile(r"coolchic_clic2024_(.+?)\.log$", re.IGNORECASE)
arm_re = re.compile(r"Using multi-region image ARM:\s*([^\s,]+)")
num_pat = r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
final_re = re.compile(
    r"Final results after quantization:\s*"
    rf"Loss:\s*{num_pat},\s*"
    rf"Rate NN:\s*{num_pat},\s*"
    rf"Rate Latent:\s*{num_pat},\s*"
    rf"Rate Img:\s*{num_pat}"
)


def parse_log(log_path: Path):
    text = log_path.read_text(encoding="utf-8", errors="replace")

    # First important section (from your example) — we require these markers.
    has_processing = "Processing image" in text
    has_mac = "Total MAC per pixel:" in text

    arm_match = arm_re.search(text)

    # Final metrics line.
    final_match = final_re.search(text)

    missing = []
    if not has_processing:
        missing.append("'Processing image' section")
    if not has_mac:
        missing.append("'Total MAC per pixel' line")
    if arm_match is None:
        missing.append("'Using multi-region image ARM' line")
    if final_match is None:
        missing.append("'Final results after quantization' line")

    if missing:
        print(f"[SKIP] {log_path}: missing {', '.join(missing)}")
        return None

    name_match = log_name_re.search(log_path.name)
    if name_match is None:
        print(f"[SKIP] {log_path}: filename does not match clic2024_<id>_job_<n>.log")
        return None

    clic_id = name_match.group(1)
    # job_id = int(name_match.group(2))
    assert arm_match is not None
    arm_config = arm_match.group(1)
    assert final_match is not None
    loss, rate_nn, rate_latent, rate_img = map(float, final_match.groups())
    return {
        "clic_id": clic_id,
        # "job_id": job_id,
        "arm_config": arm_config,
        "loss": loss,
        "rate_nn": rate_nn,
        "rate_latent": rate_latent,
        "rate_img": rate_img,
        "source": str(log_path),
    }


for run_dir in run_dirs:
    results_dir = Path(full_runs_dir) / run_dir / "results"
    print(f"\n=== {run_dir} ===")

    if not results_dir.exists():
        print(f"[WARN] Missing directory: {results_dir}")
        continue

    grouped_by_clic_id = {}
    for log_path in sorted(results_dir.glob("*.log")):
        parsed = parse_log(log_path)
        if parsed is None:
            continue

        grouped_by_clic_id.setdefault(parsed["clic_id"], []).append(parsed)

    if not grouped_by_clic_id:
        print("No valid logs found.")
        continue

    # Per clic_id group: keep the row with lowest loss, but sort groups by min job_id in group.
    rows = []
    for clic_id, group in grouped_by_clic_id.items():
        best_row = min(group, key=lambda r: r["loss"])
        # min_job_id = min(r["job_id"] for r in group)
        row = dict(best_row)
        # row["group_min_job_id"] = min_job_id
        rows.append(row)

    rows.sort(key=lambda r: r["clic_id"]) # (r["group_min_job_id"], r["clic_id"]))

    print("clic_id:10 job_id ARM_Config loss rate_nn rate_latent rate_img")
    for row in rows:
        print(
            f"{row['clic_id'][:10]:10} "
            # f"{row['job_id']:6d} "
            f"{row['arm_config']:9} "
            f"{row['loss']:.12g} "
            f"{row['rate_nn']:.12g} "
            f"{row['rate_latent']:.12g} "
            f"{row['rate_img']:.12g}"
        )

    n = len(rows)
    avg_loss = sum(r["loss"] for r in rows) / n
    avg_rate_nn = sum(r["rate_nn"] for r in rows) / n
    avg_rate_latent = sum(r["rate_latent"] for r in rows) / n
    avg_rate_img = sum(r["rate_img"] for r in rows) / n

    print(
        f"Average "
        f"{avg_loss:.12g} "
        f"{avg_rate_nn:.12g} "
        f"{avg_rate_latent:.12g} "
        f"{avg_rate_img:.12g}"
    )



=== 2026_01_15_YCoCg_two_by_two_Clic ===
[WARN] Missing directory: ../logs_cluster/logs/full_runs/february/2026_01_15_YCoCg_two_by_two_Clic/results

=== 2026_01_30_YCoCg_one_by_one_Clic ===
[WARN] Missing directory: ../logs_cluster/logs/full_runs/february/2026_01_30_YCoCg_one_by_one_Clic/results

=== 2026_01_30_YCoCg_tre_by_tre_Clic ===
[WARN] Missing directory: ../logs_cluster/logs/full_runs/february/2026_01_30_YCoCg_tre_by_tre_Clic/results

=== 2026_02_09_YCoCg_tre_by_tre_Clic ===
[SKIP] ../logs_cluster/logs/full_runs/february/2026_02_09_YCoCg_tre_by_tre_Clic/results/2026_02_09__23_54_31__coolchic_clic2024_13e9b6076516eb2c4645cac24dd7d65aed06a5d2fded98e8e44f450c0dab8393.log: missing 'Final results after quantization' line
[SKIP] ../logs_cluster/logs/full_runs/february/2026_02_09_YCoCg_tre_by_tre_Clic/results/2026_02_09__23_54_31__coolchic_clic2024_196233210c24b81c00b591a994e4e90ada677eddf0430bc191263ba3a31a2538.log: missing 'Final results after quantization' line
[SKIP] ../logs_clus